In [1]:
import numpy as np
import random

class Bank:
    def __init__(self, liquidity_ratio=0.2, liquidation_discount=0.9, return_multiplier=0.25, theta = 1, R=1.2, r = 1, equity=0):

        # caracteristicas
        self.total_deposits = 0.0
        self.alpha = liquidity_ratio      # Fracción en reservas líquidas L_0
        self.liquid_reserves = 0.0        # L_t
        self.illiquid_assets = equity     # I_t
        self.ell = liquidation_discount   # Descuento por venta forzosa (l)
        self.return_multiplier = return_multiplier
        self.theta = theta
        self.R = R
        self.r = r
        
        # estados
        self.is_solvent = True
        self.depositors = {}              # Registro contable {agent_id: balance}

    def receive_deposit(self, agent_id, amount):
        self.total_deposits += amount
        self.liquid_reserves += amount * self.alpha
        self.illiquid_assets += amount * (1.0 - self.alpha)
        self.depositors[agent_id] = self.depositors.get(agent_id, 0.0) + amount 
                                    # dinero que tenia o cero          + nuevo dinero depositado 

    def process_withdrawal(self, agent_id, amount):
        if not self.is_solvent:
            return 0.0  # El banco quebró; late arrival recibe 0 bajo FCFS

        # 1. Pago directo con reservas líquidas
        if self.liquid_reserves >= amount:
            self.liquid_reserves -= amount
            self.depositors[agent_id] = 0 # self.depositors.get(agent_id, 0.0) - amount/r
            return amount

        # 2. Liquidación forzosa de activo ilíquido
        shortfall = amount - self.liquid_reserves
        required_liquidation = shortfall / self.ell

        if self.illiquid_assets >= required_liquidation:
            paid = amount
            self.liquid_reserves = 0.0
            self.illiquid_assets -= required_liquidation
            self.depositors[agent_id] = 0 # self.depositors.get(agent_id, 0.0) - paid/r
            return paid
        else:
            # 3. Quiebra: el banco entrega todo su remanente liquidado y cierra
            paid = self.liquid_reserves + (self.illiquid_assets * self.ell)
            self.liquid_reserves = 0.0
            self.illiquid_assets = 0.0
            self.is_solvent = False
            self.depositors[agent_id] = 0 #self.depositors.get(agent_id, 0.0) - paid/r
            return paid

    def settle_maturity(self):
        # Depositantes que quedan (P)
        remaining_depositors = {aid: bal for aid, bal in self.depositors.items() if bal > 0} 

        if not self.is_solvent:
            # Limpiar balance
            for agent_id, balance in remaining_depositors.items():
                self.depositors[agent_id] = 0.0
            return 

        # Retorno Bruto del banco.
        Y_theta = 1.0 + self.return_multiplier * self.theta
        self.liquid_reserves += self.illiquid_assets * Y_theta
        self.illiquid_assets = 0

        total_remaining_claims = sum(remaining_depositors.values())
        if total_remaining_claims == 0:
            return
        
        # Pagar depositantes o pro-ratear si no alcanza
        
        if self.liquid_reserves >= total_remaining_claims * self.R:
            for agent_id, balance in remaining_depositors.items():
                promised_payout = balance * self.R
                agent_id.cash += promised_payout
                self.liquid_reserves -= promised_payout
                
                self.depositors[agent_id] = 0.0
        else:
            self.is_solvent = False
            L = self.liquid_reserves
            for agent_id, balance in remaining_depositors.items():
                share = balance / total_remaining_claims
                agent_id.cash += L * share
                self.liquid_reserves -= L * share
                
                self.depositors[agent_id] = 0.0
        
        self.liquid_reserves = 0.0 if abs(self.liquid_reserves) < 1e-10 else self.liquid_reserves
        return


class Agent:
    def __init__(self, initial_cash=1.0, is_patient=True):

        # caracteristicas
        self.cash = initial_cash
        self.deposited_money = 0.0
        self.bank = None                  # Banco asignado explícitamente
        self.is_patient = is_patient      # Tipo de preferencia de consumo
        self.signal = None                # Señal privada s_i

        # estado
        self.has_withdrawn = False

    def deposit(self, target_bank, amount=None):
        """t = 0: Asigna el banco y transfiere los fondos."""
        if amount is None:
            amount = self.cash

        if amount <= 0 or amount > self.cash:
            return

        self.cash -= amount
        self.deposited_money += amount
        self.bank = target_bank           # Guarda la identidad del banco
        self.bank.receive_deposit(self, amount)

    def decide_and_act(self, act = False):
        """
        t = 1: Decide retirar o esperar usando su banco registrado.
        """
        if self.bank is None or self.deposited_money <= 0 or self.has_withdrawn:
            return

        # Impaciente siempre retira; o revisión que indiva que debe actuar
        should_withdraw = (not self.is_patient) or act

        if should_withdraw:
            requested_amount = self.deposited_money * self.bank.r
            received_amount = self.bank.process_withdrawal(self, requested_amount)

            self.cash += received_amount
            self.deposited_money = 0.0
            self.has_withdrawn = True

    def evaluation(self):
        self.act = False # or True
        return 
   

In [47]:
class D_D:
    def __init__(self, N=10, R=1.5, r=1.28, Lambda=0.3, seed = False):

        # parámetros agentes
        self.N = N
        # parámetros banco
        self.liquidity_ratio = 0
        self.liquidation_discount = 1
        self.return_multiplier = R - 1
        self.theta = 1
        self.equity = 0
        self.Lambda = Lambda
        self.R = R
        self.r = r
        self.EW = (R-r)/(r*(R-1))
        # parámetros random
        self.seed = seed

    def create_environment(self):
        
        self.bank = Bank(liquidity_ratio=self.liquidity_ratio, liquidation_discount=self.liquidation_discount, return_multiplier=self.return_multiplier, R=self.R, r=self.r, equity=self.equity)

        if self.seed == True:
            random.seed(self.seed)
            
        impatient_indices = set(random.sample(range(self.N), round(self.Lambda * self.N)))

        self.agents = [Agent(initial_cash=1.0, is_patient=(i not in impatient_indices)) for i in range(self.N)]
        self.impatient_agents = [agent for agent in self.agents if not agent.is_patient]
        self.patient_agents =   [agent for agent in self.agents if agent.is_patient]

    def waiting_equilibrium(self, Force = False):

        # ============================================================
        # t = 0: Creation
        # ============================================================
        
        self.create_environment()
        
        for agent in self.agents:
            agent.deposit(self.bank)
        if (self.Lambda > self.EW) and (Force == False):
            print('No waiting equilibrium')
            self.withdrawing_equilibrium()
            return
        elif self.Lambda == self.EW:
            print('Unstable equilibrium')

        if Force == True:
            print('\033[1m Forced Waiting - No Equilibrium \033[0m\n')
        else:
            print('\033[1m Waiting Equilibrium \033[0m\n')
            
        print("t = 0")
        print(f"Depositors: {self.N}")
        print("--- Initial balance sheet ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {self.bank.is_solvent}")
        
        # ============================================================
        # t = 1: Sequential service
        # ============================================================
        
        print("\nt = 1")
        print(f"Impatient depositors: {round(self.Lambda * self.N)}")
        print(f"Patient depositors: {self.N - round(self.Lambda * self.N)}")
               
        random.shuffle(self.impatient_agents)
        
        for agent in self.impatient_agents:
            agent.decide_and_act()
        
        print("--- After impatient depositors withdraw ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {self.bank.is_solvent}")
        
        
        # ============================================================
        # t = 2: No Panic / Solvent
        # ============================================================
        
        print("\nNo Panic Case")
        A = self.bank.is_solvent

        print("\nt = 2")
        
        self.bank.settle_maturity()
        
        print("--- Maturity settlement ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {A}")
        
        # ============================================================
        # Payoffs
        # ============================================================
        
        print("\n--- Individual Payoffs ---")
        
        for i, agent in enumerate(self.agents):
            depositor_type = "Patient" if agent.is_patient else "Impatient"
            print(f"Agent {i}: " f"{depositor_type} | " f"Payoff = {round(agent.cash, 2)}")

    def withdrawing_equilibrium(self):

        # ============================================================
        # t = 0: Creation
        # ============================================================

        self.create_environment()

        for agent in self.agents:
            agent.deposit(self.bank)

        print('\033[1m Withdrawing Equilibrium \033[0m\n')
            
        print("t = 0")
        print(f"Depositors: {self.N}")
        print("--- Initial balance sheet ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {self.bank.is_solvent}")
        
        # ============================================================
        # t = 1: Sequential service
        # ============================================================
        
        print("\nt = 1")
        print(f"Impatient depositors: {round(self.Lambda * self.N)}")
        print(f"Patient depositors: {self.N - round(self.Lambda * self.N)}")
               
        random.shuffle(self.impatient_agents)
        
        for agent in self.impatient_agents:
            agent.decide_and_act()
        
        print("--- After impatient depositors withdraw ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {self.bank.is_solvent}")
        
        
        # ============================================================
        # t = 2: Panic / Insolvent
        # ============================================================
        
        print("\nPanic Case")
        
        for agent in self.patient_agents:
            agent.decide_and_act(act = True)
            if agent.cash < 1:
                break
        
        print("\nt = 2")
        
        self.bank.settle_maturity()
        
        print("--- No maturity settlement ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {self.bank.is_solvent}")
        
        # ============================================================
        # Payoffs
        # ============================================================
        
        print("\n--- Individual Payoffs ---")
        
        for i, agent in enumerate(self.agents):
            depositor_type = "Patient" if agent.is_patient else "Impatient"
            print(f"Agent {i}: " f"{depositor_type} | " f"Payoff = {round(agent.cash, 2)}")

    def partial_run(self, panic = 1):

        # ============================================================
        # t = 0: Creation
        # ============================================================
        
        self.create_environment()

        panic = min(max(0, int(panic)),len(self.patient_agents))
        
        for agent in self.agents:
            agent.deposit(self.bank)
        if (self.Lambda + panic/self.N > self.EW):
            print('No waiting equilibrium')
        else:
            print('Waiting equilibrium')

        print(f'\033[1m {panic} Patient(s) decide to withdraw - Partial Run\033[0m\n')
            
        print("t = 0")
        print(f"Depositors: {self.N}")
        print("--- Initial balance sheet ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {self.bank.is_solvent}")
        
        # ============================================================
        # t = 1: Sequential service
        # ============================================================
        
        print("\nt = 1")
        print(f"Impatient depositors: {round(self.Lambda * self.N)}")
        print(f"Patient depositors: {self.N - round(self.Lambda * self.N)}")
        
        
        random.shuffle(self.impatient_agents)
        
        for agent in self.impatient_agents:
            agent.decide_and_act()
        
        print("--- After impatient depositors withdraw ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {self.bank.is_solvent}")

    
        print("\nPartial Panic Case")
        print(f"Panicking patient depositors: {panic}")
        
        
        for i in range(panic):
             self.patient_agents[i].decide_and_act(act = True)
             if self.patient_agents[i].cash < 1:
                 break
                 
        print("--- After panicking patient depositor(s) withdraw ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {self.bank.is_solvent}")

        A = self.bank.is_solvent
        
        
        # ============================================================
        # t = 2: Partial Panic
        # ============================================================
        
        
        print("\nt = 2")
        
        self.bank.settle_maturity()
        
        print("--- Maturity settlement ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {A}")
        
        # ============================================================
        # Payoffs
        # ============================================================
        
        print("\n--- Individual Payoffs ---")
        
        for i, agent in enumerate(self.agents):
            depositor_type = "Patient" if agent.is_patient else "Impatient"
            print(f"Agent {i}: " f"{depositor_type} | " f"Payoff = {round(agent.cash, 2)}")

    def coordination(self, p = 0.5):

        # ============================================================
        # t = 0: Creation
        # ============================================================
        
        self.create_environment()
        
        for agent in self.agents:
            agent.deposit(self.bank)
        if (self.Lambda > self.EW):
            print('No waiting equilibrium')
            self.withdrawing_equilibrium()
            return
            
        print("t = 0")
        print(f"Depositors: {self.N}")
        print("--- Initial balance sheet ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {self.bank.is_solvent}")
        
        # ============================================================
        # t = 1: Sequential service
        # ============================================================
        
        print("\nt = 1")
        print(f"Impatient depositors: {round(self.Lambda * self.N)}")
        print(f"Patient depositors: {self.N - round(self.Lambda * self.N)}")
               
        random.shuffle(self.impatient_agents)
        
        for agent in self.impatient_agents:
            agent.decide_and_act()
        
        print("--- After impatient depositors withdraw ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {self.bank.is_solvent}")
        
        # ============================================================
        # Coordinating
        # ============================================================

        # Confusing Equilibrium
        i = 0
        for agent in self.patient_agents:
            if random.random() < p:
                agent.decide_and_act(act = True)
                i += 1

        print(f'{i} patient depositors confused and runned')
        # Not confused reevaluate and decide
        if self.Lambda + i/self.N  > self.EW:
            print('Everyone Run!')
            for agent in self.patient_agents:
                agent.decide_and_act(act = True)
                
        # ============================================================
        # t = 2: Coordination Problem
        # ============================================================
        
        print("\nCoordination Problem")
        A = self.bank.is_solvent

        print("\nt = 2")
        
        self.bank.settle_maturity()
        
        print("--- Maturity settlement ---")
        print(f"Liquid: {round(self.bank.liquid_reserves, 2)} | " f"Illiquid: {round(self.bank.illiquid_assets, 2)} | " f"Solvent: {A}")
        
        # ============================================================
        # Payoffs
        # ============================================================
        
        print("\n--- Individual Payoffs ---")
        
        for i, agent in enumerate(self.agents):
            depositor_type = "Patient" if agent.is_patient else "Impatient"
            print(f"Agent {i}: " f"{depositor_type} | " f"Payoff = {round(agent.cash, 2)}")

        if self.Lambda + i/self.N  > self.EW:
            return True
        else:
            return False

In [53]:
Simu = D_D(N=100, R=1.5, r=1.28, Lambda=0.1, seed = 1)

In [54]:
Simu.waiting_equilibrium()

 Waiting Equilibrium 

t = 0
Depositors: 100
--- Initial balance sheet ---
Liquid: 0.0 | Illiquid: 100.0 | Solvent: True

t = 1
Impatient depositors: 10
Patient depositors: 90
--- After impatient depositors withdraw ---
Liquid: 0.0 | Illiquid: 87.2 | Solvent: True

No Panic Case

t = 2
--- Maturity settlement ---
Liquid: 0.0 | Illiquid: 0 | Solvent: True

--- Individual Payoffs ---
Agent 0: Patient | Payoff = 1.45
Agent 1: Patient | Payoff = 1.45
Agent 2: Patient | Payoff = 1.45
Agent 3: Patient | Payoff = 1.45
Agent 4: Patient | Payoff = 1.45
Agent 5: Patient | Payoff = 1.45
Agent 6: Patient | Payoff = 1.45
Agent 7: Patient | Payoff = 1.45
Agent 8: Impatient | Payoff = 1.28
Agent 9: Patient | Payoff = 1.45
Agent 10: Patient | Payoff = 1.45
Agent 11: Patient | Payoff = 1.45
Agent 12: Patient | Payoff = 1.45
Agent 13: Patient | Payoff = 1.45
Agent 14: Patient | Payoff = 1.45
Agent 15: Impatient | Payoff = 1.28
Agent 16: Patient | Payoff = 1.45
Agent 17: Impatient | Payoff = 1.28
Agent 1

In [55]:
Simu.withdrawing_equilibrium()

 Withdrawing Equilibrium 

t = 0
Depositors: 100
--- Initial balance sheet ---
Liquid: 0.0 | Illiquid: 100.0 | Solvent: True

t = 1
Impatient depositors: 10
Patient depositors: 90
--- After impatient depositors withdraw ---
Liquid: 0.0 | Illiquid: 87.2 | Solvent: True

Panic Case

t = 2
--- No maturity settlement ---
Liquid: 0.0 | Illiquid: 0.0 | Solvent: False

--- Individual Payoffs ---
Agent 0: Patient | Payoff = 1.28
Agent 1: Patient | Payoff = 1.28
Agent 2: Patient | Payoff = 1.28
Agent 3: Patient | Payoff = 1.28
Agent 4: Patient | Payoff = 1.28
Agent 5: Patient | Payoff = 1.28
Agent 6: Patient | Payoff = 1.28
Agent 7: Patient | Payoff = 1.28
Agent 8: Impatient | Payoff = 1.28
Agent 9: Patient | Payoff = 1.28
Agent 10: Patient | Payoff = 1.28
Agent 11: Patient | Payoff = 1.28
Agent 12: Patient | Payoff = 1.28
Agent 13: Patient | Payoff = 1.28
Agent 14: Patient | Payoff = 1.28
Agent 15: Impatient | Payoff = 1.28
Agent 16: Patient | Payoff = 1.28
Agent 17: Impatient | Payoff = 1.28


In [56]:
Simu.partial_run(panic =1)

Waiting equilibrium
 1 Patient(s) decide to withdraw - Partial Run

t = 0
Depositors: 100
--- Initial balance sheet ---
Liquid: 0.0 | Illiquid: 100.0 | Solvent: True

t = 1
Impatient depositors: 10
Patient depositors: 90
--- After impatient depositors withdraw ---
Liquid: 0.0 | Illiquid: 87.2 | Solvent: True

Partial Panic Case
Panicking patient depositors: 1
--- After panicking patient depositor(s) withdraw ---
Liquid: 0.0 | Illiquid: 85.92 | Solvent: True

t = 2
--- Maturity settlement ---
Liquid: 0.0 | Illiquid: 0 | Solvent: True

--- Individual Payoffs ---
Agent 0: Patient | Payoff = 1.28
Agent 1: Patient | Payoff = 1.45
Agent 2: Patient | Payoff = 1.45
Agent 3: Patient | Payoff = 1.45
Agent 4: Patient | Payoff = 1.45
Agent 5: Patient | Payoff = 1.45
Agent 6: Patient | Payoff = 1.45
Agent 7: Patient | Payoff = 1.45
Agent 8: Impatient | Payoff = 1.28
Agent 9: Patient | Payoff = 1.45
Agent 10: Patient | Payoff = 1.45
Agent 11: Patient | Payoff = 1.45
Agent 12: Patient | Payoff = 1.45

In [61]:
A = Simu.coordination(p = 0.5)

t = 0
Depositors: 100
--- Initial balance sheet ---
Liquid: 0.0 | Illiquid: 100.0 | Solvent: True

t = 1
Impatient depositors: 10
Patient depositors: 90
--- After impatient depositors withdraw ---
Liquid: 0.0 | Illiquid: 87.2 | Solvent: True
34 patient depositors confused and runned
Everyone Run!

Coordination Problem

t = 2
--- Maturity settlement ---
Liquid: 0.0 | Illiquid: 0.0 | Solvent: False

--- Individual Payoffs ---
Agent 0: Patient | Payoff = 1.28
Agent 1: Patient | Payoff = 1.28
Agent 2: Patient | Payoff = 1.28
Agent 3: Patient | Payoff = 1.28
Agent 4: Patient | Payoff = 1.28
Agent 5: Patient | Payoff = 1.28
Agent 6: Patient | Payoff = 1.28
Agent 7: Patient | Payoff = 1.28
Agent 8: Impatient | Payoff = 1.28
Agent 9: Patient | Payoff = 1.28
Agent 10: Patient | Payoff = 1.28
Agent 11: Patient | Payoff = 1.28
Agent 12: Patient | Payoff = 1.28
Agent 13: Patient | Payoff = 1.28
Agent 14: Patient | Payoff = 1.28
Agent 15: Impatient | Payoff = 1.28
Agent 16: Patient | Payoff = 1.28


In [59]:
(1.5-1.28)/(1.28*0.5)

0.34374999999999994